# 실습 15. 성향점수 분석

사회약학 연구방법 노트의 '실습 15 성향점수 분석'을 따라 하는 노트북입니다. 15장의 예제 코호트(가상 청구자료의 신규 사용자 10,000명)로 성향점수를 추정하고, 1:1 매칭과 역확률 가중을 한 뒤 균형을 진단하고 1년 입원의 상대위험도를 추정합니다. 셀을 위에서부터 차례로 실행하세요. 자료는 사이트가 만든 가상 자료이며 실제 환자 자료가 아닙니다.


## 가. 실습 데이터 준비

Colab에서는 런타임이 새로 배정될 때마다 lifelines를 설치합니다(셀 18에서 씁니다).

### 셀 1. lifelines 설치

In [ ]:
!pip install lifelines

### 셀 2. 코호트 자료 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
df = pd.read_csv(BASE + "ps_cohort.csv", storage_options=UA)
print(df.shape)
df.head(3)

### 셀 3. 보정 전 두 군의 기저특성과 입원 위험

In [ ]:
covs = ["age", "female", "hf", "ckd", "prior_hosp",
        "n_drug", "tertiary"]
print(df["drugA"].value_counts())

t1 = df.groupby("drugA")[covs + ["hosp_1y"]].mean().T
t1.columns = ["drug B", "drug A"]
print("crude RR:",
      round(t1.loc["hosp_1y", "drug A"] / t1.loc["hosp_1y", "drug B"], 3))
t1.round(3)

## 나. 성향점수 추정과 겹침

결과변수 자리에 치료(drugA)를 놓은 로지스틱 회귀입니다.

### 셀 4. 성향점수 모형 (로지스틱 회귀)

In [ ]:
f_ps = ("drugA ~ age + female + hf + ckd + prior_hosp"
        " + n_drug + tertiary")
ps_fit = smf.logit(f_ps, data=df).fit(disp=0)
print(ps_fit.summary())

### 셀 5. 환자별 성향점수와 C 통계량

In [ ]:
from sklearn.metrics import roc_auc_score

df["ps"] = ps_fit.predict(df)             # 환자별 성향점수
print(df.groupby("drugA")["ps"]
        .agg(["mean", "min", "max"]).round(3))
print("C statistic:",
      round(roc_auc_score(df["drugA"], df["ps"]), 3))

new = pd.DataFrame({
    "age": [78, 58], "female": [1, 0], "hf": [1, 0],
    "ckd": [1, 0], "prior_hosp": [1, 0], "n_drug": [9, 4],
    "tertiary": [1, 0]})
print("two patients:", ps_fit.predict(new).round(3).tolist())

### 셀 6. 두 군의 성향점수 분포와 겹침

In [ ]:
a = df["drugA"] == 1                      # A 사용자면 True
bins = np.linspace(0, 1, 41)              # 0.025 간격의 막대
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.hist(df.loc[~a, "ps"], bins=bins, density=True,
        alpha=0.5, label="Drug B")
ax.hist(df.loc[a, "ps"], bins=bins, density=True,
        alpha=0.5, label="Drug A")
ax.set_xlabel("Propensity score (probability of drug A)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()

ps_lo = max(df.loc[a, "ps"].min(), df.loc[~a, "ps"].min())
ps_hi = min(df.loc[a, "ps"].max(), df.loc[~a, "ps"].max())
outside = (df["ps"] < ps_lo) | (df["ps"] > ps_hi)
print(f"common range {ps_lo:.3f}-{ps_hi:.3f},",
      "outside:", outside.sum())

## 다. 매칭과 가중

로짓 성향점수로 짝을 짓고, 같은 성향점수로 가중치를 만듭니다.

### 셀 7. 로짓 성향점수와 caliper

In [ ]:
df["logit"] = np.log(df["ps"] / (1 - df["ps"]))   # 로짓 성향점수
sd = df["logit"].std()
caliper = 0.2 * sd                        # 허용 한계
print("SD of logit PS:", round(sd, 3))
print("caliper       :", round(caliper, 3))

### 셀 8. 1:1 최근접 매칭 함수

In [ ]:
def match_1to1(logit, treat, caliper, seed=7):
    """1:1 최근접 매칭 (비복원, caliper 적용)."""
    rng = np.random.default_rng(seed)
    a_rows = np.where(treat == 1)[0]      # A 사용자의 행 번호
    b_rows = np.where(treat == 0)[0]      # B 사용자의 행 번호
    b_logit = logit[b_rows].copy()        # 아직 남은 B의 로짓
    pairs = []
    for i in rng.permutation(a_rows):     # A를 무작위 순서로
        dist = np.abs(b_logit - logit[i])     # 모든 B와의 거리
        j = np.argmin(dist)                   # 가장 가까운 B
        if dist[j] <= caliper:                # 한계 안이면
            pairs.append((i, b_rows[j]))      # 짝으로 기록
            b_logit[j] = np.inf               # 쓴 B는 지움
    return pd.DataFrame(pairs, columns=["a_row", "b_row"])

pairs = match_1to1(df["logit"].to_numpy(),
                   df["drugA"].to_numpy(), caliper)
print(len(pairs), "pairs")
pairs.head(3)

### 셀 9. 매칭 코호트 만들기

In [ ]:
ma = df.iloc[pairs["a_row"]].copy()       # 짝을 찾은 A 사용자
mb = df.iloc[pairs["b_row"]].copy()       # 짝이 된 B 사용자
ma["pair"] = range(len(pairs))            # 짝 번호 0, 1, 2, ...
mb["pair"] = range(len(pairs))
m = pd.concat([ma, mb])                   # 매칭 코호트 (긴 형식)

gap = np.abs(ma["logit"].to_numpy() - mb["logit"].to_numpy())
print("matched A:", len(ma), "of", a.sum(),
      f"({len(ma) / a.sum():.1%})")
print("largest distance in a pair:", round(gap.max(), 3))
show = ["pair", "id", "drugA", "age", "hf", "ps"]
m.sort_values(["pair", "drugA"])[show].head(4).round(3)

### 셀 10. 짝을 못 찾은 A 사용자는 누구인가

In [ ]:
df["matched"] = df["id"].isin(m["id"]).astype(int)
um = df[a].groupby("matched")[["ps", "age", "hf",
                               "hosp_1y"]].mean()
um.insert(0, "n", df[a]["matched"].value_counts())
print("unused B users:", ((~a) & (df["matched"] == 0)).sum())
um.round(3)

### 셀 11. 역확률 가중치와 안정화 가중치

In [ ]:
pA = df["drugA"].mean()                   # A를 받은 비율 0.4053
ps = df["ps"]
df["w"] = np.where(a, 1 / ps, 1 / (1 - ps))            # IPTW
df["sw"] = np.where(a, pA / ps, (1 - pA) / (1 - ps))   # 안정화

print(df[["w", "sw"]].agg(["mean", "min", "max"]).round(3))
print(df.groupby("drugA")[["w", "sw"]].sum().round(0))

wb = np.linspace(0, 6, 61)                # 0.1 간격의 막대
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(df.loc[~a, "sw"], bins=wb, alpha=0.5, label="Drug B")
ax.hist(df.loc[a, "sw"], bins=wb, alpha=0.5, label="Drug A")
ax.set_xlabel("Stabilized weight")
ax.set_ylabel("Number of patients")
ax.legend()
plt.tight_layout()

## 라. 균형 진단과 효과 추정

균형(SMD)을 먼저 확인한 다음 결과를 비교합니다.

### 셀 12. 표준화 평균차(SMD) 함수

In [ ]:
def smd(x, treat, w=None):
    """표준화 평균차. w를 주면 가중 평균과 가중 분산으로."""
    x = np.asarray(x, dtype=float)
    t = np.asarray(treat) == 1
    w = np.ones(len(x)) if w is None else np.asarray(w)
    m1 = np.average(x[t], weights=w[t])           # A군 평균
    m0 = np.average(x[~t], weights=w[~t])         # B군 평균
    v1 = np.average((x[t] - m1) ** 2, weights=w[t])     # 분산
    v0 = np.average((x[~t] - m0) ** 2, weights=w[~t])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)

print("hf, before:", round(smd(df["hf"], df["drugA"]), 3))

### 셀 13. 보정 전, 매칭 뒤, 가중 뒤의 균형 표

In [ ]:
bal = pd.DataFrame({
    "before": [smd(df[v], df["drugA"]) for v in covs],
    "matched": [smd(m[v], m["drugA"]) for v in covs],
    "IPTW": [smd(df[v], df["drugA"], df["w"]) for v in covs],
}, index=covs).abs()
print(bal.max().round(3))
bal.round(3)

### 셀 14. Love plot

In [ ]:
y = np.arange(len(covs))
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for col, mk in [("before", "o"), ("matched", "s"), ("IPTW", "^")]:
    ax.scatter(bal[col], y, marker=mk, label=col)
ax.axvline(0.1, ls="--", color="gray")    # 기준선 0.1
ax.set_yticks(y)
ax.set_yticklabels(covs)
ax.invert_yaxis()                         # 첫 변수를 맨 위에
ax.set_xlabel("Absolute standardized mean difference")
ax.legend()
plt.tight_layout()

### 셀 15. 세 분석의 입원 위험과 상대위험도

In [ ]:
def risks(d, w=None):
    """A군 위험, B군 위험, 상대위험도, 위험차 (w는 가중치 열 이름)."""
    t = d["drugA"] == 1
    w = np.ones(len(d)) if w is None else d[w].to_numpy()
    rA = np.average(d.loc[t, "hosp_1y"], weights=w[t])
    rB = np.average(d.loc[~t, "hosp_1y"], weights=w[~t])
    return [rA, rB, rA / rB, rA - rB]

est = pd.DataFrame(
    {"crude": risks(df), "matched": risks(m), "IPTW": risks(df, "w")},
    index=["risk A", "risk B", "RR", "RD"]).T
est.round(4)

### 셀 16. 매칭 코호트의 상대위험도와 강건 신뢰구간

In [ ]:
def gee_fit(formula, data, groups, weights=None, family=None):
    """독립 작업상관 GEE. 기본은 포아송(상대위험도)."""
    if family is None:
        family = sm.families.Poisson()
    return smf.gee(formula, groups=groups, data=data,
                   family=family, weights=weights,
                   cov_struct=sm.cov_struct.Independence()).fit()

gm = gee_fit("hosp_1y ~ drugA", m, "pair")
print(gm.summary().tables[1])
print("clusters:", gm.model.num_group,
      " cov type:", gm.cov_type)
ci = gm.conf_int().loc["drugA"]
print("RR:", round(np.exp(gm.params["drugA"]), 3),
      " 95% CI:", np.exp(ci).round(3).tolist())

### 셀 17. 세 분석의 상대위험도와 매칭 코호트의 위험차

In [ ]:
def rr_ci(g):
    """GEE 결과에서 RR과 95% 신뢰구간을 꺼냅니다."""
    lo, hi = g.conf_int().loc["drugA"]
    return np.exp([g.params["drugA"], lo, hi])

g0 = gee_fit("hosp_1y ~ drugA", df, "id")             # 보정 전
gw = gee_fit("hosp_1y ~ drugA", df, "id", df["sw"])   # IPTW
rr = pd.DataFrame([rr_ci(g0), rr_ci(gm), rr_ci(gw)],
                  index=["crude", "matched", "IPTW"],
                  columns=["RR", "lower", "upper"])
print(rr.round(3))

gd = gee_fit("hosp_1y ~ drugA", m, "pair",
             family=sm.families.Gaussian())           # 위험차
lo, hi = 100 * gd.conf_int().loc["drugA"]
print("matched RD (%p):", round(100 * gd.params["drugA"], 1),
      f"({lo:.1f}, {hi:.1f})")

### 셀 18. 입원까지의 시간으로 본 위험비 (Cox 모형)

In [ ]:
from lifelines import CoxPHFitter

cols = ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"]
c1 = CoxPHFitter().fit(df, "hosp_day", "hosp_1y", formula="drugA")
c2 = CoxPHFitter().fit(m, "hosp_day", "hosp_1y", formula="drugA",
                       cluster_col="pair")
c3 = CoxPHFitter().fit(df, "hosp_day", "hosp_1y", formula="drugA",
                       weights_col="sw", robust=True)
hr = pd.concat([c.summary[cols] for c in (c1, c2, c3)])
hr.index = ["crude", "matched", "IPTW"]
hr.columns = ["HR", "lower", "upper"]
hr.round(3)

### 셀 19. 측정하지 않은 변수와 음성 대조 결과

In [ ]:
fr = [smd(df["frail"], df["drugA"]), smd(m["frail"], m["drugA"]),
      smd(df["frail"], df["drugA"], df["w"])]
print("frailty SMD (before, matched, IPTW):", np.round(fr, 2))
print(m.groupby("drugA")[["frail", "injury_ed"]].mean().round(3))

nc = gee_fit("injury_ed ~ drugA", m, "pair")     # 음성 대조 결과
print("negative control RR:", rr_ci(nc).round(2))

## 과제 정답

**과제 1.** 매칭 코호트의 상대위험도와 그 신뢰구간에 대한 E-value를 구합니다. 보정 전 상대위험도의 E-value와 비교합니다.

### 셀 20. 과제 1 정답. E-value

In [ ]:
def evalue(rr):
    """E-value. RR이 1보다 작으면 역수를 먼저 취합니다."""
    rr = 1 / rr if rr < 1 else rr
    return rr + np.sqrt(rr * (rr - 1))

r = rr.loc["matched"]                     # RR, lower, upper
print("point estimate:", round(evalue(r["RR"]), 2))
print("rounded 0.91  :", round(evalue(0.91), 2))
covers_1 = r["lower"] <= 1 <= r["upper"]
print("CI limit      :",
      1.0 if covers_1 else round(evalue(r["upper"]), 2))
c0 = rr.loc["crude"]
print("crude         :", round(evalue(c0["RR"]), 2),
      round(evalue(c0["lower"]), 2))

**과제 2.** caliper를 로짓 성향점수 표준편차의 0.05, 0.1, 0.2, 0.5, 1.0배와 caliper 없음으로 바꿔 매칭 쌍 수, 가장 큰 SMD, 상대위험도를 표로 만듭니다.

### 셀 21. 과제 2 정답. caliper 폭 바꾸기

In [ ]:
lg = df["logit"].to_numpy()
tr = df["drugA"].to_numpy()
rows = []
for k in [0.05, 0.1, 0.2, 0.5, 1.0, np.inf]:
    p = match_1to1(lg, tr, k * sd)
    mk = pd.concat([df.iloc[p["a_row"]], df.iloc[p["b_row"]]])
    worst = max(abs(smd(mk[v], mk["drugA"])) for v in covs)
    rows.append([k, len(p), round(len(p) / a.sum(), 3),
                 round(worst, 3), round(risks(mk)[2], 3)])
pd.DataFrame(rows, columns=["caliper (SD)", "pairs",
                            "share of A", "max SMD", "RR"])

**과제 3.** 성향점수 모형에서 심부전(hf)을 빼고 안정화 가중치를 다시 만들어, 가중 뒤 심부전의 SMD와 상대위험도가 어떻게 달라지는지 봅니다.

### 셀 22. 과제 3 정답. 심부전을 뺀 성향점수 모형

In [ ]:
f2 = "drugA ~ age + female + ckd + prior_hosp + n_drug + tertiary"
ps2 = smf.logit(f2, data=df).fit(disp=0).predict(df)
df["sw2"] = np.where(a, pA / ps2, (1 - pA) / (1 - ps2))

for name, w in [("full model", "sw"), ("without hf", "sw2")]:
    print(f"hf SMD, {name}: {smd(df['hf'], df['drugA'], df[w]):.3f}")
g2 = gee_fit("hosp_1y ~ drugA", df, "id", df["sw2"])
print("IPTW RR, without hf:", rr_ci(g2).round(3))
print("IPTW RR, full model:", rr.loc["IPTW"].to_numpy().round(3))